# RASCAR V16 — воспроизводимый финальный ранжировщик

Распакуйте архив целиком. Этот ноутбук запускает **CPU replay из сохранённых предсказаний и признаков**, поверх замороженного V15 (LB 0.92822). Он не переобучает исходный V15 и не запускает GPU-модели заново. Все необходимые для replay входы лежат внутри папки `rascar_v16`.

Новый LB ещё неизвестен. OOF: **0.98126458**; без правила, работающего только на OOF, **0.97828792**, при этом тестовый CSV одинаков. Валидация: fit 25 → predict 26 и fit 26 → predict 25.

При необходимости установите зависимости в отдельное окружение: `python -m pip install -r requirements-replay.txt`.


In [ ]:
from pathlib import Path
import sys, subprocess, json, hashlib
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'replay.py').exists():
    ROOT = ROOT / 'rascar_v16'
assert (ROOT / 'release_manifest.json').is_file(), 'Run from the extracted directory'
manifest = json.loads((ROOT / 'release_manifest.json').read_text())
ROOT


In [ ]:
result = subprocess.run([sys.executable, str(ROOT / 'replay.py')], cwd=ROOT,
                        check=True, capture_output=True, text=True)
print(result.stdout)


In [ ]:
names = ['baseline_lb092822', 'final_targeted_short',
         'final_targeted_short_with_contradiction', 'final_targeted_with_contradiction']
rows = [json.loads((ROOT / 'results' / f'{name}.json').read_text()) for name in names]
pd.DataFrame(rows)[['experiment', 'pooled_ap', 'day_25', 'day_26',
                    'ap_person', 'ap_nude_hands', 'ap_cup', 'ap_tray']]


In [ ]:
name = manifest['main']
submission_path = ROOT / 'results' / 'submissions' / f'submission_{name}.csv'
submission = pd.read_csv(submission_path)
assert list(submission.columns) == ['id', 'score']
assert len(submission) == 362 and submission.id.is_unique
assert submission.score.between(0, 1).all()
assert hashlib.sha256(submission_path.read_bytes()).hexdigest() == manifest['release_experiments'][name]['sha256']
print('Submission:', submission_path.resolve())
submission.head()


## Просмотр ошибок и остальные эксперименты

Откройте `REPORT.html`: сравнение с базой, все сохранённые варианты и 79 примеров проблемных рамок. `selected_test_audit.csv` содержит все 362 строки; `visual_review_log.csv` — журнал повторного просмотра изменённых рамок; `validation_changed_cases.csv` — существенные изменения OOF. На тесте нет официальных меток, визуальная оценка использовалась при разработке.

Основной файл для загрузки: `submission_v16_targeted_short.csv`. Абляция без нового правила высоты: `submission_v16_targeted_conservative.csv`.

`fit_dino_heads.py` заново обучает головы LogisticRegression/CatBoost на сохранённых DINO-эмбеддингах. Для повторения всех метамоделей также есть `pose_features.py` и `evaluate_wevis.py`. Запускайте их в копии папки, затем `replay.py`: результаты сохраняются в `results/`.

Исходные GPU-скрипты `build_dino_features.py`, `extract_multiview_pose.py`, `extract_wevis.py` сохранены как provenance исходного Kaggle-запуска вместе с `prepare_remote.py`, launcher-ячейками, логами и ревизиями моделей. Они требуют Kaggle-окружение, GPU и доступ к весам, а не запускаются этим CPU-ноутбуком. SAM3 недоступен (HTTP 401) и в итоговом решении не использован.
